# SchoolGraph 단독 질의와 검증 (B08~B10)

공통 Neo4j의 실제 데이터(선정 쪽수 전체: 청크 182 · 승인 행동 42)로 `rag.school.run()`을 실행하고 검증한다.

| 부품 | 수업 근거 | 여기서 |
|---|---|---|
| S01 전문검색 | 38-1 by_text | `chunk_text`(cjk) 검색 + HAS_CHUNK·allowed 필터 |
| S02 출처 보존 | 38-2 with_source | 검색 결과가 처음부터 document_id·chunk_id·쪽수·원문을 들고 다님 |
| S03 벡터·하이브리드 | 38-1 by_vector, 38-2 Hybrid | text-embedding-3-small 벡터 + 순위 결합(RRF) |
| S04 그래프 확장 | 38-3 EXPAND | 청크 → 행동 → 역할(basis)·개념, 비교 질문만 다른 출처 확장 |
| S05·S06 고정 도구·흐름 | 37-6 rows/@tool, 38-5 agent | 매개변수 Cypher + 고정 순서 LangGraph(+역할별 보조 검색), 실행 추적 |
| S07 평가 | 38-2 top_articles, 38-5 ask/is_correct | 정답 청크 Hit@5·MRR, 상태, 금지 내용, DB 대조 |

Finding은 두 가지다: (1) 승인된 **학교 행동**(SchoolAction, 지금은 시나리오 표에만 있음), (2) 검색된 **원문 인용**(행동 노드가 없는 청크의 원문을 그대로 옮김). 둘 다 LLM 없이 DB 내용만으로 만든다.

**평가용 4문항(S-A2·S-F2·S-T2·S-G2)은 결과를 따로 보고하고, 이 문항들로 규칙을 조정하지 않는다**(골든셋 7장).

In [1]:
import json
import re
import sys
from collections import Counter, defaultdict
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from index.school.dictionaries import load_dictionaries
from rag.school import run
from rag.school.graph import default_deps
from rag.school.search import search_school_chunks
from rag.school.validation_local import verify_against_db

deps = default_deps()
deps["trace"] = []
DRIVER, DATABASE = deps["driver"], deps["database"]
BOOKS = load_dictionaries()
ALLOWED = ["moe_school_manual_v3", "gyeonggi_school_guide_2025", "kdca_measles_2026",
           "kdca_diagnosis_reporting_2026", "moleg_infectious_disease_act_21521"]   # golden S-Q1 QueryInput
PREFERRED = ["moe_school_manual_v3", "gyeonggi_school_guide_2025"]
EVAL_ONLY = {"S-A2", "S-F2", "S-T2", "S-G2"}
pd.set_option("display.max_colwidth", 110)

golden = (ROOT / "data" / "specs" / "golden" / "school.md").read_text(encoding="utf-8")
QUESTIONS = {qid: (q, region, status) for qid, q, region, status in re.findall(
    r"^\| (S-[A-Z]\d) \| (.+?) \| (GYEONGGI|NATIONAL) \| ([A-Z_]+)", golden, re.M)}
EXPECTED = defaultdict(set)
for qids, chunk in re.findall(r"^\| C\d\d \| ([^|]+) \| [a-z0-9_]+ \| ([a-z0-9_]+:p\d{4}:[a-z0-9]+) \|", golden, re.M):
    for qid in re.findall(r"S-[A-Z]\d", qids):
        EXPECTED[qid].add(chunk)
records, _, _ = DRIVER.execute_query("MATCH (c:Chunk) RETURN c.chunk_id AS id", database_=DATABASE)
in_db = {r["id"] for r in records}
cited = set().union(*EXPECTED.values())
print(f"골든 질문 {len(QUESTIONS)}개 / 골든 Citation 청크 {len(cited)}개 중 DB에 있음 {len(cited & in_db)}개")

골든 질문 14개 / 골든 Citation 청크 31개 중 DB에 있음 31개


## 1. 검색 품질 (S03·S07) - 정답 청크가 상위 5위 안에 있나

골든셋 5장에서 질문별 정답 청크를 모았다(정답 청크가 없는 G 문항 제외). Hit@5 = 상위 5위 안에 정답 청크가 하나라도 있는 질문 비율, MRR = 첫 정답 순위의 역수 평균.

In [2]:
rows = []
for mode in ("fulltext", "vector", "hybrid"):
    for qid, chunks in sorted(EXPECTED.items()):
        hits = search_school_chunks(DRIVER, {"question": QUESTIONS[qid][0], "allowed_document_ids": ALLOWED,
                                             "preferred_document_ids": PREFERRED}, BOOKS,
                                    embedder=deps["embedder"], mode=mode, k=10, database=DATABASE)["hits"]
        ranks = [h["rank"] for h in hits if h["chunk_id"] in chunks]
        rows.append({"방식": mode, "질문": qid, "평가용": qid in EVAL_ONLY, "첫 정답 순위": min(ranks) if ranks else None})
ranking = pd.DataFrame(rows)
metric = ranking.groupby(["방식", "평가용"]).agg(
    문항=("질문", "count"),
    Hit5=("첫 정답 순위", lambda s: f"{int((s.fillna(99) <= 5).sum())}/{len(s)}"),
    MRR=("첫 정답 순위", lambda s: round((1 / s.dropna()).sum() / len(s), 3)))
display(metric)
display(ranking.pivot(index="질문", columns="방식", values="첫 정답 순위"))

문항 Hit5    MRR
방식       평가용                  
fulltext False   9  5/9  0.425
         True    3  2/3  0.444
hybrid   False   9  5/9  0.511
         True    3  2/3  0.500
vector   False   9  6/9  0.528
         True    3  2/3  0.283

방식,fulltext,hybrid,vector
질문,,,
S-A1,1.0,1.0,1.0
S-A2,1.0,1.0,2.0
S-A3,8.0,NaN,NaN
S-A4,1.0,1.0,4.0
S-F1,6.0,7.0,1.0
S-F2,3.0,2.0,4.0
S-Q1,5.0,1.0,1.0
S-Q2,NaN,NaN,NaN
S-Q3,3.0,8.0,NaN


## 2. 골든 질문 실행

모든 문항에 고정하는 **안전 검사**(실패하면 assert): 계약 구조(run 안), Neo4j 대조, '반드시·의무' 단정 없음, S-Q1에 다른 단계 내용(NEIS 보고·능동감시·보건소 신고) 없음, 근거 없음 문항의 빈 결과. 기대 status 일치와 정답 청크 인용 수는 **품질 지표**로 보고한다.

In [3]:
FORBIDDEN_IN_SQ1 = ("NEIS", "나이스", "능동감시", "보건소")
results, table = {}, []
for qid, (question, region, golden_status) in QUESTIONS.items():
    deps["trace"].clear()
    query = {"question": question, "region": region, "allowed_document_ids": ALLOWED,
             "preferred_document_ids": PREFERRED}
    out = run(query, deps=deps)
    report = verify_against_db(out, DRIVER, DATABASE)
    claims = " ".join(f["claim"] for f in out["findings"])
    cited_chunks = {c["chunk_id"] for c in out["citations"]}
    results[qid] = {"query": query, "output": out, "trace": list(deps["trace"])}
    table.append({"ID": qid, "평가용": qid in EVAL_ONLY, "status": out["status"], "골든 기대": golden_status,
                  "status 일치": out["status"] == golden_status,
                  "정답 청크 인용": f"{len(cited_chunks & EXPECTED[qid])}/{len(EXPECTED[qid])}" if EXPECTED[qid] else "-",
                  "Finding": len(out["findings"]),
                  "행동/인용": f"{sum(bool(f['action_ids']) for f in out['findings'])}/"
                               f"{sum(not f['action_ids'] for f in out['findings'])}",
                  "DB 대조": not report["bad_citations"] and not report["bad_findings"],
                  "의무 단정 없음": not re.search("반드시|의무", claims),
                  "S-Q1 단계 섞임 없음": qid != "S-Q1" or not any(w in claims for w in FORBIDDEN_IN_SQ1)})
summary = pd.DataFrame(table)
display(summary)
print("status 일치:", int(summary["status 일치"].sum()), "/", len(summary),
      "(개발용", int(summary[~summary["평가용"]]["status 일치"].sum()), "/", int((~summary["평가용"]).sum()),
      "· 평가용", int(summary[summary["평가용"]]["status 일치"].sum()), "/", int(summary["평가용"].sum()), ")")
assert summary["DB 대조"].all() and summary["의무 단정 없음"].all() and summary["S-Q1 단계 섞임 없음"].all()
for qid in ("S-G1", "S-G2"):
    assert results[qid]["output"]["status"] == "INSUFFICIENT_EVIDENCE" and not results[qid]["output"]["findings"]

,ID,평가용,status,골든 기대,status 일치,정답 청크 인용,Finding,행동/인용,DB 대조,의무 단정 없음,S-Q1 단계 섞임 없음
0,S-Q1,False,ANSWERED,ANSWERED,True,3/9,16,12/4,True,True,True
1,S-Q2,False,ANSWERED,ANSWERED,True,0/6,3,0/3,True,True,True
2,S-Q3,False,PARTIAL,PARTIAL,True,1/3,4,0/4,True,True,True
3,S-Q4,False,ANSWERED,ANSWERED,True,3/4,12,6/6,True,True,True
4,S-A1,False,ANSWERED,ANSWERED,True,3/3,8,0/8,True,True,True
5,S-A2,True,ANSWERED,ANSWERED,True,1/1,8,0/8,True,True,True
6,S-A3,False,ANSWERED,ANSWERED,True,0/3,2,0/2,True,True,True
7,S-A4,False,ANSWERED,ANSWERED,True,3/4,7,0/7,True,True,True
8,S-F1,False,ANSWERED,ANSWERED,True,2/3,4,0/4,True,True,True
9,S-F2,True,ANSWERED,ANSWERED,True,0/1,3,0/3,True,True,True


status 일치: 14 / 14 (개발용 10 / 10 · 평가용 4 / 4 )


## 3. S-Q1 결과 보기 (Finding·Citation·warnings·실행 추적)

In [4]:
q1 = results["S-Q1"]
print("status:", q1["output"]["status"])
for w in q1["output"]["warnings"]:
    print("warning:", w)
display(pd.DataFrame([{"종류": "행동" if f["action_ids"] else "원문 인용", "claim": f["claim"],
                       "chunk": f["chunk_ids"][0].split(":", 1)[1], "정답 청크": f["chunk_ids"][0] in EXPECTED["S-Q1"]}
                      for f in q1["output"]["findings"]]))
display(pd.DataFrame([{"step": t["step"], **{k: (v if not isinstance(v, list) or len(v) < 6 else f"{len(v)}개")
                                             for k, v in t.items() if k not in ("step", "terms", "focus")}}
                      for t in q1["trace"]]))

status: ANSWERED


,종류,claim,chunk,정답 청크
0,원문 인용,[경기도 가이드북 PDF 166쪽 · 원문 배치: 대응 제1단계 · 원문이 제시한 담당 예시: 담임교사] 원문: “담임교사|대응 제1단계|1. 감염병 유증상자 발견 - 보건교사에게 연락(마...,p0166:t01r01,True
1,원문 인용,"[교육부 매뉴얼 PDF 57쪽 · 원문 배치: 대응 제1단계] 원문: “ 보호자에게 의료기관 진료 여부를 확인하고 미진료 시 진료를 받도록 하며, 이때 출결 관련 필요 서류 및 학교 홈페이...",p0057:b01,False
2,원문 인용,[교육부 매뉴얼 PDF 54쪽 · 원문 배치: 대응 제1단계] 원문: “ 감염병 (의심)환자가 교내에서 혼자 이동하는 것은 원칙적으로 금지  이동 시에는 담임(담당) 교사가 동행하며 2m...,p0054:b01,True
3,원문 인용,[교육부 매뉴얼 PDF 57쪽 · 원문 배치: 대응 제1단계] 원문: “2) 보건(담당)교사  감염병 확진 인지 시 즉시 대응 제2단계 활동 수행  위생수칙 교육 등 기타 활동은 학교 안...,p0057:b02,False
4,행동,"[교육부 매뉴얼 PDF 36쪽, 모의훈련 시나리오 예시 · 대응 제1단계] 원문이 제시한 담당 예시: 보건교사 — 교사와 진한이가 보건실에 도착한 즉시 손을 씻도록 요청한다. (시점: 보건실...",p0036:t01r02,True
5,행동,"[교육부 매뉴얼 PDF 36쪽, 모의훈련 시나리오 예시 · 대응 제1단계] 원문이 제시한 담당 예시: 보건교사 — 마스크를 착용한다.",p0036:t01r02,True
6,행동,"[교육부 매뉴얼 PDF 36쪽, 모의훈련 시나리오 예시 · 대응 제1단계] 원문이 제시한 담당 예시: 보건교사 — 체온을 측정한다.",p0036:t01r02,True
7,행동,"[교육부 매뉴얼 PDF 36쪽, 모의훈련 시나리오 예시 · 대응 제1단계] 원문이 제시한 담당 예시: 보건교사 — 환자의 증상과 징후를 관찰하고 감염병이 의심되는지를 확인한다.",p0036:t01r02,True
8,행동,"[교육부 매뉴얼 PDF 36쪽, 모의훈련 시나리오 예시 · 대응 제1단계] 원문이 제시한 담당 예시: 보건교사 — 진한이가 의료기관 진료를 받지 않았음을 확인한다.",p0036:t01r02,True
9,행동,"[교육부 매뉴얼 PDF 36쪽, 모의훈련 시나리오 예시 · 대응 제1단계] 원문이 제시한 담당 예시: 보건교사 — 담임교사에게 보호자에게 연락하도록 요청한다. (조건: 진한이가 의료기관 진료...",p0036:t01r02,True


,step,roles,stages,compare,broad,legal_duty,out_of_scope,mode,chunk_ids,role,added,found,kept,actions,findings,quote_chunks,bad_citations,bad_findings
0,plan,"[HOMEROOM_TEACHER, HEALTH_TEACHER]",[RESPONSE_1],False,True,False,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,search,NaN,NaN,NaN,NaN,NaN,NaN,hybrid,8개,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,role_search,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HOMEROOM_TEACHER,[moe_school_manual_v3:p0065:t01r02],NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,role_search,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HEALTH_TEACHER,"[moe_school_manual_v3:p0057:b02, moe_school_manual_v3:p0036:t01r02]",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.0,12.0,NaN,NaN,NaN,NaN,NaN
5,expand,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12개,NaN,NaN,NaN,NaN
6,compose,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,"[gyeonggi_school_guide_2025:p0166:t01r01, moe_school_manual_v3:p0057:b01, moe_school_manual_v3:p0054:b01, ...",NaN,NaN
7,verify,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0


## 4. 실패해야 하는 경우 (T6·allowed·문서 유출)

- T6: 존재하지 않는 chunk_id·action_id를 가진 출력은 구조 검증은 통과해도 Neo4j 대조에서 걸려야 한다.
- `allowed_document_ids=[]`이면 결과가 없어야 한다.
- 경기도 문서만 허용하면 교육부 청크가 Finding·Citation에 하나도 없어야 한다(보조 검색·확장 포함).

In [5]:
forged = json.loads(json.dumps(q1["output"]))
forged["citations"][0]["chunk_id"] = "moe_school_manual_v3:p0036:t09r09"
forged["findings"][0]["chunk_ids"] = ["moe_school_manual_v3:p0036:t09r09"]
action_finding = next(i for i, f in enumerate(forged["findings"]) if f["action_ids"])
forged["findings"][action_finding]["action_ids"] = ["moe_school_manual_v3:p0036:a99"]
t6 = verify_against_db(forged, DRIVER, DATABASE)
print("T6 가짜 청크 검출:", list(t6["bad_citations"]), "/ 문제 Finding:", sorted(t6["bad_findings"]))
assert t6["bad_citations"] and action_finding in t6["bad_findings"]

empty = run({"question": QUESTIONS["S-Q1"][0], "region": "GYEONGGI", "allowed_document_ids": []}, deps=deps)
print("allowed=[]:", empty["status"], empty["warnings"])
assert empty["status"] == "INSUFFICIENT_EVIDENCE" and not empty["citations"]
for qid in ("S-Q1", "S-Q4", "S-A1"):
    only_gg = run({"question": QUESTIONS[qid][0], "region": "GYEONGGI",
                   "allowed_document_ids": ["gyeonggi_school_guide_2025"]}, deps=deps)
    docs = {c["document_id"] for c in only_gg["citations"]} | {f["chunk_ids"][0].split(":")[0] for f in only_gg["findings"]}
    print(qid, "경기도만 허용:", only_gg["status"], docs)
    assert docs <= {"gyeonggi_school_guide_2025"}

T6 가짜 청크 검출: ['moe_school_manual_v3:p0036:t09r09'] / 문제 Finding: [0, 4]
allowed=[]: INSUFFICIENT_EVIDENCE ['허용된 문서가 없다(allowed_document_ids=[]).']


S-Q1 경기도만 허용: ANSWERED {'gyeonggi_school_guide_2025'}


S-Q4 경기도만 허용: ANSWERED {'gyeonggi_school_guide_2025'}


S-A1 경기도만 허용: ANSWERED {'gyeonggi_school_guide_2025'}


## 5. 증빙 저장

질문별 QueryInput/Output·실행 추적·결과 표·검색 순위를 Git 제외 폴더에 남긴다. Citation 검수표는 사람이 PDF와 대조할 때 쓴다.

In [6]:
out_dir = ROOT / "data" / "interim" / "school" / datetime.now(UTC).strftime("query_%Y%m%dT%H%M%S_%fZ")
out_dir.mkdir(parents=True)
(out_dir / "results.json").write_text(json.dumps(results, ensure_ascii=False, indent=1, allow_nan=False), encoding="utf-8")
summary.to_csv(out_dir / "summary.csv", index=False, encoding="utf-8-sig")
ranking.to_csv(out_dir / "search_ranking.csv", index=False, encoding="utf-8-sig")
review = pd.DataFrame([{**c, "정답 청크": c["chunk_id"] in EXPECTED["S-Q1"], "검수: PDF 쪽 일치": "",
                        "검수: 발췌가 주장을 지지": "", "메모": ""} for c in q1["output"]["citations"]])
review.to_csv(out_dir / "citation_review_S-Q1.csv", index=False, encoding="utf-8-sig")
print("저장:", out_dir.relative_to(ROOT), sorted(p.name for p in out_dir.iterdir()))
DRIVER.close()

저장: data\interim\school\query_20261104T075824_293600Z ['citation_review_S-Q1.csv', 'results.json', 'search_ranking.csv', 'summary.csv']
